In [1]:
import sys
sys.path.append("C:\\Users\\jy97geri\\pyrpl_updated\\pyrpl") # replace with your pyrpl path
import pyrpl

In [ ]:
from pyrpl import Pyrpl

HOSTNAME = "10.203.128.57"

# Initialize Pyrpl and connect to Red Pitaya
p = Pyrpl(config="iir_module",
          hostname=HOSTNAME,
          reloadfpga=True,
          reloadserver=True,
          #gui=True
          # Set to False for notebook use
         )

# print(f"✓ Connected to Red Pitaya at {HOSTNAME}")
# print(f"Available modules: {list(p.rp.modules.keys())}")

INFO:pyrpl:All your PyRPL settings will be saved to the config file
    C:\Users\jy97geri\pyrpl_user_dir\config\iir_module.yml
If you would like to restart PyRPL with these settings, type "pyrpl.exe iir_module" in a windows terminal or 
    from pyrpl import Pyrpl
    p = Pyrpl('iir_module')
in a python terminal.
INFO:pyrpl.redpitaya:Successfully connected to Redpitaya with hostname 10.203.128.57.


: 

In [ ]:
# Cell 2: Load optimal 8th-order Butterworth filter (loops=400)
import numpy as np
import scipy.signal as signal
import matplotlib.pyplot as plt

# Get IIR module from Red Pitaya (correct PyRPL access)
iir = p.rp.iir

# Filter specifications
cutoff_hz = 1000
order = 8
loops = 400      # OPTIMAL: balance between stability and bandwidth
gain = 1  # CRITICAL: Extremely small gain for 8th-order filter (start conservative)

# CORRECT poles for PyRPL: Only positive imaginary parts, in Hz (not rad/s)!
# These are s-domain poles: |z| = 0.980-0.996 (all stable!)
poles_pyrpl = [
    -0.1951 + 0.9808j,  # Pole pair 1: |z|=0.996 (acceptable)
    -0.5556 + 0.8315j,  # Pole pair 2: |z|=0.989 (safe)
    -0.8315 + 0.5556j,  # Pole pair 3: |z|=0.983 (safe)
    -0.9808 + 0.1951j,  # Pole pair 4: |z|=0.980 (very safe)
  
]
# PyRPL will automatically create conjugate pairs (negative imaginary parts)

zeros_pyrpl = []  # No zeros for Butterworth

print("=" * 70)
print("8TH-ORDER BUTTERWORTH LOW-PASS FILTER (OPTIMAL - LOOPS=400)")
print("=" * 70)
print(f"Cutoff frequency: {cutoff_hz} Hz")
print(f"Filter order: {order}")
print(f"Decimation (loops): {loops}")
print(f"Effective sampling: {125e6/loops/1e3:.1f} kHz")
print(f"Gain: {gain}")
print("\nPoles for PyRPL (Hz, positive imaginary only):")
for i, pole in enumerate(poles_pyrpl, 1):
    print(f"  Pole pair {i}: {pole.real:+.2f} {pole.imag:+.2f}j Hz")
print("\nPyRPL will auto-create conjugate pairs (8 poles total)")
print("Expected z-domain: |z| = 0.996, 0.989, 0.983, 0.980 (all stable ✓)")
print("=" * 70)

# Load filter to FPGA
print("\nLoading filter to FPGA...")
try:
    with iir.do_setup:
        iir.loops = loops              # Set decimation first
        iir.poles = poles_pyrpl        # Poles in Hz (positive imag only!)
        iir.zeros = zeros_pyrpl        # No zeros for Butterworth
        iir.gain = gain                # Very small gain
        iir.input = 'in1'              # Input from ADC channel 1
        iir.output_direct = 'off' 
        iir.on = True
        iir.bypass = False     # Don't bypass
    
    print("✓ Filter loaded successfully!")
    print(f"\nLoaded configuration:")
    print(f"  Loops: {iir.loops}")
    print(f"  Number of pole pairs: {len(iir.poles)}")
    print(f"  Total poles (with conjugates): {len(iir.poles) * 2}")
    print(f"  Gain: {iir.gain}")
    print(f"  Input: {iir.input}")
    print(f"  Output direct: {iir.output_direct}")
    
    # Check for saturation warnings
    print("\n✓ Check above for saturation warnings")
    print("  If you see ANY warnings, reduce gain further:")
    print("    - Try 0.0000001 (1e-7)")
    print("    - Or even 0.00000001 (1e-8)")
    print("  The gain will be very small due to 8th-order pole product!")
    
except Exception as e:
    print(f"✗ Error loading filter: {e}")
    import traceback
    traceback.print_exc()
    print("\nTroubleshooting:")
    print("  - Poles must be in Hz, not rad/s")
    print("  - Only provide poles with POSITIVE imaginary parts")
    print("  - PyRPL auto-creates conjugate pairs")

8TH-ORDER BUTTERWORTH LOW-PASS FILTER (OPTIMAL - LOOPS=400)
Cutoff frequency: 1000 Hz
Filter order: 8
Decimation (loops): 400
Effective sampling: 312.5 kHz
Gain: 1

Poles for PyRPL (Hz, positive imaginary only):
  Pole pair 1: -0.20 +0.98j Hz
  Pole pair 2: -0.56 +0.83j Hz
  Pole pair 3: -0.83 +0.56j Hz
  Pole pair 4: -0.98 +0.20j Hz

PyRPL will auto-create conjugate pairs (8 poles total)
Expected z-domain: |z| = 0.996, 0.989, 0.983, 0.980 (all stable ✓)

Loading filter to FPGA...
✓ Filter loaded successfully!

Loaded configuration:
  Loops: 400
  Number of pole pairs: 4
  Total poles (with conjugates): 8
  Gain: 1.0
  Input: in1
  Output direct: off

✓ Check above for saturation warnings
  If you see ANY warnings, reduce gain further:
    - Try 0.0000001 (1e-7)
    - Or even 0.00000001 (1e-8)
  The gain will be very small due to 8th-order pole product!


: 